In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as scio
import joblib
import torch

In [ ]:
def pre_process_infer(data,delta_t,scaler):
  #organize the data
  x= data[:,:,0]
  z= data[:,:,1]
  dx= np.zeros_like(x) #zeros_like returns an np array with the same shape and size as the original array, but with zeros
  dz= np.zeros_like(z)

  dx[:,1:]= x[:,1:]-x[:,:-1] #performs subtraction on data(2,:)-data(1,:), data(3,:)-data(2,:), ... data(end,:)-data(end-1,:)
  dz[:,1:]= z[:,1:]-z[:,:-1]

  vx= dx/delta_t
  vz= dz/delta_t

  ax= np.zeros_like(x)
  az= np.zeros_like(z)

  #starts from the second index as acceleration requires velocity at two time points to begin
  ax[:,2:]= (vx[:,2:]-vx[:,1:-1])/delta_t
  az[:,2:]= (vz[:,2:]-vz[:,1:-1])/delta_t

  dt= np.full_like(x,delta_t)  #full_like(size of which array to copy, value to fill)
  feat= np.concatenate((x[:,:,None],z[:,:,None],vx[:,:,None],vz[:,:,None],ax[:,:,None],az[:,:,None],dt[:,:,None],dx[:,:,None],dz[:,:,None]),axis=2)

  #normalize the data
  physical_width,physical_height= 15,15
  tracks= torch.from_numpy(feat).float()
  #divide the tracks into two different features - one to the LSTM and the other to the ST-GATE
  lstm_feat= tracks[:,:,[0,1,2,3,4,5]]
  stg_feat= tracks[:,:,[6,7,8]]
  #the LSTM-features would be scaled by z-score
  tracksn= scaler.transform(lstm_feat.reshape(-1,lstm_feat.shape[-1]))
  lstm_feat_scaled= torch.tensor(tracksn.reshape(lstm_feat.shape),dtype=torch.float32)
  #the ST-GATE features would be scaled with a reference
  stg_feat[:,:,0]/= delta_t #0.02 #divide dt by a reference time
  stg_feat[:,:,1]/= physical_width #divide dx by a reference width
  stg_feat[:,:,2]/= physical_height #divide dz by a reference height

  st_ode_feat= torch.cat((lstm_feat_scaled,stg_feat),axis=2) #concatenate the normalized features

  return st_ode_feat

In [ ]:
device = "cpu"
#torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import torch
import torch.nn as nn

class STGate(nn.Module):
  def __init__(self,hidden_dim):
    super(STGate,self).__init__()
    self.mlp= nn.Sequential(
        nn.Linear(3,32),
        nn.ReLU(),
        nn.Linear(32,hidden_dim)
    )

  def forward(self,st_input):
    #st_input= [dx,dy,dz] dim:Bx3
    u= torch.torch.exp(-torch.relu(self.mlp(st_input)))

    return u

In [ ]:
#Now, let's introduce the ST-Gate and add latent ODE
import torch.nn as nn

class STencoder(nn.Module):
  def __init__(self,input_dim=10,hidden_dim=128):
    super().__init__()
    self.hidden_dim= hidden_dim

    self.lstm_cell= nn.LSTMCell(6,hidden_dim)
    self.st_gate= STGate(hidden_dim)

  def forward(self,packed_seq,return_seq=False):
    data, batch_sizes,sorted_idx,unsorted_idx= packed_seq

    B= batch_sizes[0]

    h = torch.zeros(B, self.hidden_dim, device=data.device)
    c = torch.zeros(B, self.hidden_dim, device=data.device)

    outputs = []
    ptr = 0

    for t, bs in enumerate(batch_sizes):
      # ---- slice packed data ----
      step_data = data[ptr:ptr + bs]
      ptr += bs

      lstm_in = step_data[:, :6]
      stg_in  = step_data[:, 7:10]

      h_active = h[:bs]
      c_active = c[:bs]

      # ---- LSTM update ----
      h_lstm, c_new = self.lstm_cell(lstm_in, (h_active, c_active))

      # ---- ST-Gate ----
      u = self.st_gate(stg_in)

      # ---- gated update ----
      h_new = u * h_active + (1 - u) * h_lstm

      # ---- write back ----
      h = h.clone()
      c = c.clone()
      h[:bs] = h_new
      c[:bs] = c_new

      if return_seq:
        outputs.append(h_new)

    #so the reason why we are packing again is that we aren't using ODE(enc)->z->ODE(dec) which would be a sequence to vector model where the model learns to reconstruct a trajectory at each point from a latent sequence that is given. since we still haven't gone to an ODE yet, we'd be using a sequence-to-sequence model where the decoder tries to reconstruct a sequence given a sequence - for the ST-Gate only case - return_seq Flag is set to True
    if return_seq:
      packed_out = PackedSequence(
            torch.cat(outputs, dim=0),
            batch_sizes,
            sorted_idx,
            unsorted_idx
        )
      return packed_out
    else:
      return h[unsorted_idx]

In [ ]:
#add the latent ODE
from torchdiffeq import odeint
class LatentODE(nn.Module):
  def __init__(self,hidden_dim):
    super().__init__()
    self.ode_func= nn.Sequential(
        nn.Linear(hidden_dim,hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim,hidden_dim)
    )
  # This forward method now defines the derivative function f(t,y) for odeint.
  # 't' is the current time point (scalar from odeint), 'y' is the current latent state (tensor).
  # The ODE is autonomous (does not explicitly depend on 't'), so 't' is not used here for calculation.
  def forward(self,t,y):
    return self.ode_func(y) # Returns dy/dt

In [ ]:
class Decoder(nn.Module):
    def __init__(self, hidden_dim, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, h):
        """
        h: Tensor of shape [T, B, hidden_dim] or [B, hidden_dim]
        """
        return self.net(h)

In [ ]:
def predict_next(model, packed_seq, x_input, dt=1.0):
    """
    Predict x(t + dt) from history
    """
    with torch.no_grad():
        # Encode history
        h0 = model.encoder(packed_seq) #encode the physics of the given trajectory

        # Integrate ONE step
        t = torch.tensor([0.0, dt], device=h0.device)
        h_next = odeint(model.latent, h0, t)[-1] #integrate it over a time step

        # Decode
        x_next = model.decoder(h_next)

    return x_next

In [ ]:
import torch.nn as nn

class latentModule(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=128, out_dim=2):
        super().__init__()

        self.encoder = STencoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim
        )

        self.latent= LatentODE(hidden_dim)

        self.decoder = Decoder(
            hidden_dim=hidden_dim,
            out_dim=out_dim
        )

    def forward(self, input, packed_input):
        """
        packed_input: PackedSequence of shape (..., input_dim)
        """
        # Encode sequence -> latent initial state
        h0 = self.encoder(packed_input)

        #Integrate latent ODE
        # Extract 1D time points up to T_max from the first trajectory in the batch.
        # All trajectories in a batch share the same time points (input_features[0, :, 6]).
        # Use .squeeze() to ensure it's a 1D tensor of shape (T_max,).
        
        time_points = input[0, :, 6].squeeze()
        #print(f"DEBUG: Shape of time_points before odeint: {time_points.shape}") # Diagnostic print

        # Integrate latent ODE
        # h_traj will have shape (num_time_points, B, hidden_dim)
        h_traj = odeint(self.latent, h0, time_points)

        # Decode the latent trajectory
        # output_decoded will have shape (num_time_points, B, out_dim)
        output_decoded = self.decoder(h_traj)

        return output_decoded

In [ ]:
class Track:
  def __init__(self,track_id,init_pos,init_frame):
    self.id= track_id
    self.position= [init_pos] #list of [x,z]
    self.frames= [init_frame]
    self.disp= 0 #disappearance counter
    self.state= 'alive'

  def length(self):
    return len(self.position)

  def last_pos(self):
    return self.position[-1]

  def add(self,pos,frame):
    self.position.append(pos)
    self.frames.append(frame)
    self.disp=0

  def mark_missed(self):
    self.disp+= 1

In [ ]:
from scipy.optimize import linear_sum_assignment

def hungarian_assign(tracks,detections,max_dist):
  #tracks would be the positions of the microbubble in frame i-1
  #detections would be the positions of the microbubble in frame i
  #our goal is to match the positions from frame i-1 to the ones in frame i
  if len(tracks)==0 or len(detections)==0:
    #if there are no positions to be matched or detections to be carried forward - like in the last frame, you are more likely to not have any future detections
    return [], list(range(len(tracks))), list(range(len(detections)))

  cost= np.zeros((len(tracks),len(detections)))
  #print(f"{cost.shape}")

  for i,trck in enumerate(tracks):
    for j,det in enumerate(detections):
      cost[i,j]= np.linalg.norm(trck.last_pos()-det)

  cost[cost>max_dist]= 1e6 #assign a huge cost to distances that exceed a certain limit
  row_ind, col_ind = linear_sum_assignment(cost)

  matches = []

  unmatched_tracks = set(range(len(tracks)))
  unmatched_dets = set(range(len(detections)))

  for r, c in zip(row_ind, col_ind):
    if cost[r, c] < 1e6:
      matches.append((r, c))
      unmatched_tracks.remove(r)
      unmatched_dets.remove(c)

  return matches, list(unmatched_tracks), list(unmatched_dets)

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence,pad_packed_sequence, PackedSequence

def likelihood_assign(tracks,detections,model,scaler,max_dist,delta_t):
  #delta_t= 0.02 #0.0025 #using this captures a very small fragment of the time frame - the model learns the vector flow, so it is able to predict even for a different time step
  dc= 1.0 #critical distance
  #print(delta_t)
  #print(max_dist)

  if len(tracks)==0 or len(detections)==0:
    return [], list(range(len(tracks))), list(range(len(detections)))

  #L= np.full((len(tracks),len(detections)),1e6) #create an mxn array filled with infinity
  model.eval()
  with torch.no_grad():
    locs=[]
    tracker=[] #stores the indices that were carried forward - so that you don't end up rewriting the index - as locs now stores index of the new array rather than the original one - for example, if track 1 was skipped and track 2 was taken, for assignment, it would be track 1, but originally, it was track 2
    for i,trk in enumerate(tracks):
      #if trk.length()<3:
      if trk.length()<5:  
        continue
      else:
        #traj= np.array(trk.position[-3:])
        traj= np.array(trk.position[-5:])
        locs.append(traj)
        tracker.append(i)

    locs= np.array(locs)

    #if not locs:
    #return [], list(range(len(tracks))), list(range(len(detections)))
    
    #first, move it to your actual scale [0,15]
    locsn = locs.copy()
    #need to change this for every example
    xmin,xmax= -9.93,9.96
    zmin,zmax= 10.03,29.99
    
    xs,zs= 15/(xmax-xmin),15/(zmax-zmin)
    
    locsn[:,:,0]= (locs[:,:,0]-xmin)*xs
    locsn[:,:,1]= (locs[:,:,1]-zmin)*zs

    #perform pre-processing and normalization
    feat= pre_process_infer(locsn,delta_t,scaler)
    B,T,_= feat.shape
    t= (torch.arange(T)*delta_t).unsqueeze(0).expand(B,-1)
    t= t.unsqueeze(-1) #make it a 3D tensor

    #get the features redy for feeding into the model
    model_feat= torch.cat((feat[:,:,:6],t,feat[:,:,6:]), dim=2)
    lengths= torch.tensor([seq.shape[0] for seq in model_feat])
    packed_input= pack_padded_sequence(model_feat,lengths,batch_first=True,enforce_sorted=False)
    x_next= predict_next(model,packed_input,model_feat[:,-1,:],dt=delta_t)

    #denormalize to actual scale
    mu= torch.tensor(scaler.mean_[:2], device=feat.device, dtype=feat.dtype)
    sd=  torch.tensor(scaler.scale_[:2], device=feat.device, dtype=feat.dtype)
    x_nextu= (x_next*sd+mu).detach().numpy()
    
    x_nextun= x_nextu.copy()
    x_nextun[:,0]= (x_nextu[:,0]/xs)+xmin
    x_nextun[:,1]= (x_nextu[:,1]/zs)+zmin

    dist_mx= np.linalg.norm(x_nextun[:, np.newaxis, :] - detections[np.newaxis, :, :],axis=2)
    dist_cost= dist_mx/max_dist #normalize the distance cost by the maximum distance to keep it in the same scale as the direction cost for better tuning

    #include a penalty for direction - as cosine similarity between displacement vectors and cosine similarity between velocities give the same angle
    last_pos= locs[:,-1,:2] #last position of the trajectory - this is the position that we are trying to match to the detections - keep this on the original scale
    pred_vec= x_nextun-last_pos

    cand_vec= detections[np.newaxis,:,:]-last_pos[:,np.newaxis,:] #candidates to all detections

    norm_pred= np.linalg.norm(pred_vec,axis=1,keepdims=True)
    norm_cand= np.linalg.norm(cand_vec,axis=2)
    cos_sim= (np.einsum('bi,bji->bj', pred_vec, cand_vec)/(norm_pred*norm_cand+1e-6)) #cosine similarity between the predicted vector and candidate vectors

    #avoid random direction and noise, which would make cosine_similarity meaningless - if the speed is too low, then we don't want to trust the direction as much - so we can set a threshold for the speed and if it's below that threshold, we can set the direction cost to 0.5 (which is the cost for random direction) - this way, we are not trusting the direction at all when the speed is low
    speed= np.linalg.norm(pred_vec,axis=1)
    #print(np.median(speed))
    slow_mask= speed<0.08#this threshold can be tuned based on the distribution of speeds in the data - you can set it to a value that captures the lower 10-20% of speeds, for example
    cos_sim[slow_mask]= 1 #if the speed is too low, set the cosine similarity to 1, which means we are not trusting the direction at all

    cos_sim= np.clip(cos_sim,-1.0,1.0) #clip the cosine similarity to avoid extreme values that can cause issues with the cost calculation

    dir_cost= (1-cos_sim)/2 #convert cosine similarity to a cost - range from 0 to 1
    L= 0.7*dist_cost+0.3*dir_cost #combine distance and direction into a single cost metric

    L[dist_mx>max_dist]=1e6

    #print("matrix shape: ", L.shape)
    # ---- Hungarian on negative likelihood ----
    row_ind, col_ind = linear_sum_assignment(L)

    matches = []
    unmatched_tracks = set(range(len(tracks)))
    unmatched_dets = set(range(len(detections)))

    for r, c in zip(row_ind, col_ind):
        d= dist_mx[r,c] #use the distance matrix for gating as normalized matrix is dimensionless and will be meaningless for gating
        if d>=1e6:
            continue #if it ended up making an impossible detection, skip it
        #gate - sigmoid score based on distance
        score= 1/(1+np.exp(d - dc))
        #print(score)
        if score>=0.6:
            #high confidence, trust enough to match
            #matches.append((r, c))
            matches.append((tracker[r],c)) #the detections won't change as we aren't skipping anything
            #unmatched_tracks.remove(r)
            unmatched_tracks.remove(tracker[r])
            unmatched_dets.remove(c)

    return matches, list(unmatched_tracks), list(unmatched_dets)

In [ ]:
def track_sequence(detections,model,scaler,max_dist=1.0,max_disp=6.0,tl_min=5.0): #tl_min=3.0
  '''args:
  detections - positions of microbubble in frame i
  max_dist - maximum distance between two microbubbles which determines if the cost can be considered
  max_disp - number of frames you have to wait before stopping the tracking of the microbubble
  tl_min - minimum microbubble points required before moving to the NN model'''

  finished= []
  active_tracks=[]
  next_id= 0
    
  #print(max_disp)
  #change this for every example
  vmax= 140.1761
  fr= 50
  dmax= vmax/fr
  vmed= 11.2335
    
  for t, det in detections.items():
    #print(t)
    det= np.asarray(det)
    #active_tracks= [trk for trk in tracks if trk.state=='alive']

    track_s = [trk for trk in active_tracks if trk.length() < tl_min] #accumulates tracks that need to go to hungarian for assignment
    track_l = [trk for trk in active_tracks if trk.length() >= tl_min]
    #print(f"length track_s: ,{len(track_s)}")
    #print(f"length track_l: ,{len(track_l)}")

    matches= []
    ut, ud= [], list(range(len(det))) #unassigned tracks, unassigned detections
    #print("ud before assignment:", len(ud))

    if len(track_l)>0 and len(ud)>0:
      m_l, ut_l, ud_l = likelihood_assign(track_l, det[ud], model, scaler, max_dist=dmax,delta_t=1/fr) 
      #print(len(m_l))

      matches += [(track_l[i], ud[j]) for i, j in m_l]

      ud= [ud[j] for j in ud_l] #update ud after the NN- assignment
      #print("ud after assignment: ", len(ud))

    u_l= [trk for trk in track_l if trk not in [m[0] for m in matches]]
    #rescue= track_s+u_l
    rescue= track_s #don't take the long unmatched tracks to Hungarian - they might be creating some noise

    if len(rescue)>0 and len(ud)>0:
      m_s, ut_s, ud_s = hungarian_assign(rescue, det[ud], max_dist=2*vmed/fr)
      #print(f"{ut}")
      #print(f"ud after hungarian: ,{ud_s}")
      matches += [(rescue[i], ud[j]) for i, j in m_s]

      #if int(t) != 0: #if it is not the first frame, else you can just use ud as such
      ud = [ud[j] for j in ud_s] #update ud after the global assignment
      #print(f"ud after hungarian: ,{len(ud)}")

    # Update matched tracks

    for trk, j in matches:
      trk.add(det[j], t)

    still_active = []
    # Handle unmatched tracks
    for trk in active_tracks:
      if trk not in [m[0] for m in matches]:
       trk.mark_missed()
       # Check for termination
       if trk.disp >= max_disp:
          trk.state = 'end'
          finished.append(trk)
       else:
        still_active.append(trk)
      else:
        still_active.append(trk)

    # Spawn new tracks
    for j in ud: #these are the ones that weren't matched
      still_active.append(Track(next_id, det[j], t))
      next_id += 1

    active_tracks = still_active

    #if int(t)>=5:
    #  break

  all_tracks = {trk.id: trk for trk in finished + active_tracks}

  return list(all_tracks.values())

In [ ]:
import os
print(f"Files in this folder: {os.listdir('.')}")
print(f"Python is currently looking in: {os.getcwd()}")

In [ ]:
#data_path = os.path.join('..', 'BUFF/simu2/', 'locs.mat')
mat = scio.loadmat('BUFF/simu2/locs.mat', struct_as_record=False, squeeze_me=True)
keys = [key for key in mat.keys() if not key.startswith("__")]
num_frames = mat[keys[0]].shape[0]

mb_locations = {}
for i in range(num_frames):
  raw_pos = mat[keys[0]][i]
  # Ensure raw_pos is always a 2D array (N, 2)
  if raw_pos.ndim == 1: # If it's a single point, raw_pos will be (2,)
    raw_pos_2d = raw_pos.reshape(1, -1) # Reshape (2,) to (1, 2)
  else:
    raw_pos_2d = raw_pos # It's already (N, 2)

    pos = np.zeros_like(raw_pos_2d) # Now pos will always be (N, 2)

    pos[:, 0] = raw_pos_2d[:, 0]*1000 #the data is in m, convert it to mm
    pos[:, 1] = raw_pos_2d[:, 1]*1000
    mb_locations[str(i)] = pos

print(f"X range: {mb_locations['0'][:,0].min():.2f} to {mb_locations['0'][:,0].max():.2f}")
print(f"Z range: {mb_locations['0'][:,1].min():.2f} to {mb_locations['0'][:,1].max():.2f}")

In [ ]:
model= latentModule()
model.load_state_dict(torch.load('main/model_seq2vec_best.pth', map_location=torch.device('cpu')))
model.to(device)
scaler= joblib.load('main/standard_scaler_adaptive_global.pkl')
final_tracks = track_sequence(mb_locations,model,scaler,max_disp=1)

In [ ]:
#check the distribution of step sizes in pixels
pX, pZ = 10, 10    # in mm
dx, dz = 0.0391, 0.0391  # 1 pixel = 0.0391 mm
all_steps_px = []
for trk in final_tracks:
    if trk.length() < 2:
        continue
    coords = np.array(trk.position)
    z_px = (coords[:,1] - pZ) / dz
    x_px = (coords[:,0] - (-pX)) / dx
    pts = np.stack([z_px, x_px], axis=1)
    steps = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    all_steps_px.extend(steps.tolist())

all_steps_px = np.array(all_steps_px)
print(f"Median step:      {np.median(all_steps_px):.1f} px")
print(f"75th percentile:  {np.percentile(all_steps_px, 75):.1f} px")
print(f"90th percentile:  {np.percentile(all_steps_px, 90):.1f} px")
print(f"95th percentile:  {np.percentile(all_steps_px, 95):.1f} px")
print(f"99th percentile:  {np.percentile(all_steps_px, 99):.1f} px")

In [ ]:
# Parameters from MATLAB code - directly accumulate without interpolation
size_z, size_x = 512, 512
min_length = 5
im_final_pred = np.zeros((size_z, size_x))

pX, pZ = 10,10 #simu2: 15, 47 in mm
dx, dz = 0.0391,0.0391 #simu2: 0.0587,0.1174  # 1 pixel = 0.0391 mm

for trk in final_tracks:
    if trk.length() <= min_length:
        continue
    
    coords = np.array(trk.position)
    
    # Convert to pixels first
    z_px = np.round((coords[:,1] - pZ) / dz).astype(int)
    x_px = np.round((coords[:,0] - (-pX)) / dx).astype(int)
    
    for z, x in zip(z_px, x_px):
        if 0 <= z < size_z and 0 <= x < size_x:
            im_final_pred[z, x] += 1

# Display
disp = im_final_pred**0.5
p99 = np.percentile(disp[disp>0], 99)
plt.figure(figsize=(10,10))
plt.imshow(disp, cmap='hot', aspect='equal',
           extent=[-10, 10, 30, 10],
           vmin=0, vmax=p99)
plt.colorbar()
plt.xlabel('Width [mm]')
plt.ylabel('Depth [mm]')
plt.title('NODE SR (Direct Accumulation)')
plt.savefig('node_direct.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#visualize the ground truth tracks
mat = scio.loadmat('BUFF/simu2/tracks.mat', struct_as_record=False, squeeze_me=True)
keys = [key for key in mat.keys() if not key.startswith("__")]
num_tracks = mat[keys[0]].shape[0]

gt_tracks = {}
for i in range(num_tracks):
  raw_pos = mat[keys[0]][i]
  # Ensure raw_pos is always a 2D array (N, 2)
  if raw_pos.ndim == 1: # If it's a single point, raw_pos will be (2,)
    raw_pos_2d = raw_pos.reshape(1, -1) # Reshape (2,) to (1, 2)
  else:
    raw_pos_2d = raw_pos # It's already (N, 2)

    pos = np.zeros_like(raw_pos_2d) # Now pos will always be (N, 2)

    pos[:, 0] = raw_pos_2d[:, 0]
    pos[:, 1] = raw_pos_2d[:, 1]
    gt_tracks[str(i)] = pos

In [ ]:
#plot the ground truth

# Parameters from MATLAB code - directly accumulate without interpolation
size_z, size_x = 512, 512
min_length = 5
im_final = np.zeros((size_z, size_x))

pX, pZ = 10,10 #simu2: 15, 47 in mm
dx, dz = 0.0391,0.0391 #simu2: 0.0587,0.1174  # 1 pixel = 0.0391 mm

for key, trk in gt_tracks.items():
    if len(trk) <= min_length:
        continue
    
    coords = trk
    
    # Convert to pixels first
    z_px = np.round((coords[:,1] - pZ) / dz).astype(int)
    x_px = np.round((coords[:,0] - (-pX)) / dx).astype(int)
    
    for z, x in zip(z_px, x_px):
        if 0 <= z < size_z and 0 <= x < size_x:
            im_final[z, x] += 1

# Display
disp = im_final**0.5
p99 = np.percentile(disp[disp>0], 99)
plt.figure(figsize=(10,10))
plt.imshow(disp, cmap='hot', aspect='equal',
           extent=[-10, 10, 30, 10],
           vmin=0, vmax=p99)
plt.colorbar()
plt.xlabel('Width [mm]')
plt.ylabel('Depth [mm]')
plt.title('GT SR (Direct Accumulation)')
plt.savefig('gt_direct.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from skimage.metrics import structural_similarity as ssim

In [ ]:
from skimage.metrics import structural_similarity as ssim
import numpy as np

def compute_ssim(im1, im2):
    # Normalize both to [0,1] using 99th percentile
    def norm99(im):
        p99 = np.percentile(im[im>0], 99) if (im>0).any() else 1
        return np.clip(im / p99, 0, 1)
    
    im1n = norm99(im1)
    im2n = norm99(im2)
    
    score = ssim(im1n, im2n, data_range=1.0)
    return score

# Your results
ssim_node     = compute_ssim(im_final, im_final_pred)
ssim_hungarian = compute_ssim(im_final, hk)

print(f"NODE SSIM:              {ssim_node:.4f}")
print(f"Hungarian-Kalman SSIM: {ssim_hungarian:.4f}")
print(f"Improvement:           {ssim_node - ssim_hungarian:.4f}")